# 08. Mini-Capstone: an Automated Processing Workflow
**Engineering Data Processing with Python** | one-day course | approx. 60 minutes

> **EXERCISE NOTEBOOK.** Work top to bottom. Run every cell in order (Shift+Enter). Where you see `____` replace it with your answer. Where you see `# TODO`, write the code. If an error mentions `____` (for example `NameError: name '____' is not defined`), you have a blank left to fill. Blanks are four underscores.


In [ ]:
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = "https://raw.githubusercontent.com/buildyourai/ei-python-data-processing/main/data.zip"   # course data (fictional / public)

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

## The brief

> **From:** Operations Manager, Kilbrack Engineering
> **Subject:** Galway September numbers
>
> Galway runs a different maintenance system from Cork. Their September 2026 export is attached
> (`data/capstone/wo_galway_2026_09.csv`). I need, by Friday:
>
> 1. A clean dataset I can trust, and a list of anything you had to exclude or that Galway needs to fix.
> 2. Work orders by type, mean time to restore for breakdowns by asset type, and the five assets with the most
>    breakdowns.
> 3. A short note on what you did and anything I should be careful about.
>
> I'll want the same every month, so please don't do it by hand.

You have everything you need from Modules 01 to 07. This notebook gives you **less scaffolding** on purpose:
the steps are listed, and there are hints and Gemini prompts, but you write most of the code.

### What "done" looks like

| # | Deliverable | Check |
|---|---|---|
| 1 | `outputs/galway_2026_09/` created by **running the pipeline with a config**, not by hand-editing data | Folder has clean data, KPI-ready data, issues, report, log |
| 2 | `config_galway.json` | Only the config differs from the Cork run, not `pipeline.py` |
| 3 | `kpis.xlsx` in the run folder | Three sheets: by type, time to restore by asset type, top 5 assets |
| 4 | Data note (markdown cell at the end) | Says what was excluded, why, and what you assumed |

Suggested timing: 10 min looking at the data, 15 min config and runs, 20 min KPIs, 10 min data note, 5 min spare.

## 0. Get the pipeline into this session

Each Colab notebook runs on its own machine, so the `pipeline.py` you wrote in Module 07 is not here.
You have two options:

* **Use your own:** download `pipeline.py` from the Module 07 session (Files panel, right-click, Download) and
  upload it here (Files panel, upload icon), then run the next cell.
* **Use the reference version:** just run the next cell. If there is no `pipeline.py` here, it copies the trainer's.

In [ ]:
import shutil, json, sys, importlib
from pathlib import Path

for name in ["pipeline.py", "config_cmms.json"]:
    if not Path(name).exists():
        shutil.copy(f"data/checkpoints/{name}", name)
        print(f"Using reference {name}")
    else:
        print(f"Using your own {name}")

import pipeline
importlib.reload(pipeline)
with open("config_cmms.json", encoding="utf-8") as f:
    cmms_config = json.load(f)
print("Pipeline and Cork config loaded.")

## 1. Look at the raw export first  [Level 2]

Before writing any config, find out what you are dealing with. Answer these in your head (or in a markdown cell):

* What separates the columns?
* What is the decimal separator for hours and cost?
* What are the column names, and which of ours does each one correspond to?
* What date formats appear in `Opened`, `Closed` and `Modified`?
* What work order type words are used?

Hint: open the file as plain text first (`open(...).readlines()[:4]`). It shows you the raw characters before
pandas interprets anything.

In [ ]:
RAW = "data/capstone/wo_galway_2026_09.csv"

# TODO: print the first 4 lines of the file as plain text


# TODO: load it with pd.read_csv (with the right sep, and dtype=str so nothing is converted yet) and look at it


# TODO: value_counts() of the type column; look at a few values from each date column

## 2. Write the Galway config  [Level 2]

Start from a copy of the Cork config and change only what is different. Everything you found in step 1 goes here.
Leave `type_map` alone for now. You will see why in step 3.

Format codes reminder: `%Y-%m-%dT%H:%M:%S` matches `2026-09-25T12:00:00`.

In [ ]:
galway = json.loads(json.dumps(cmms_config))      # independent copy
galway["run_name"] = "galway_2026_09"

# TODO: set galway["source"]["path"], ["sep"], ["decimal"]
# TODO: set galway["source"]["date_formats"] (two formats) and ["modified_formats"]
# TODO: set galway["source"]["column_map"] = {"WorkOrder": "wo_id", "AssetTag": ..., ...}  (all 12 columns)

with open("config_galway.json", "w", encoding="utf-8") as f:
    json.dump(galway, f, indent=2, ensure_ascii=False)
print(json.dumps(galway["source"], indent=2))

## 3. Run it, and let it fail properly

In [ ]:
!{sys.executable} pipeline.py config_galway.json

It should stop with **Unmapped work order types**, listing the new words. That is the pipeline doing its job:
it refuses to guess what `Reactive` means. You decide.

### [Level 1: fill the blanks]  Add the new words to the Galway type map, save, run again

In [ ]:
galway["type_map"]["PPM"] = "____"
galway["type_map"]["PLANNED MAINT."] = "____"
galway["type_map"]["____"] = "CM"
with open("config_galway.json", "w", encoding="utf-8") as f:
    json.dump(galway, f, indent=2, ensure_ascii=False)

!{sys.executable} pipeline.py config_galway.json

In [ ]:
run_dir = Path("outputs/galway_2026_09")
files = {p.name for p in run_dir.iterdir()} if run_dir.exists() else set()
report = pd.read_csv(run_dir / "dq_report.csv") if (run_dir / "dq_report.csv").exists() else None
check({"work_orders_clean.csv", "work_orders_kpi_ready.csv", "dq_issues.csv", "dq_report.csv", "pipeline.log"} <= files,
      "Galway processed by config alone. pipeline.py did not change.",
      "Outputs missing. Read the error from the run above: it names the setting to fix.")
report

## 4. Read your own quality report  [Level 2]

Open `dq_issues.csv` and look at the error rows. For each error rule that fired, decide: is it a data entry
problem Galway should fix, or something you need to ask about? (For example: is 300 hours a typo for 3.00, or
300 minutes? You cannot know. That is why it goes back to Galway rather than being "fixed" by you.)

In [ ]:
# TODO: load dq_issues.csv from run_dir and show the rows where dq_error is True,
#       with wo_id, asset_id, wo_type, date_raised, date_closed, labour_hours, parts_cost_eur, dq_issues

## 5. KPIs  [Level 3: write it yourself, Gemini allowed]

Using **only KPI-ready rows** (`work_orders_kpi_ready.csv`):

* **(a) Work orders by type**: count of PM, CM, INSP.
* **(b) Mean time to restore for breakdowns by asset type**: for **closed** CM work orders,
  hours from `date_raised` to `date_closed`. Show count, mean and median per `asset_type`.
* **(c) Top 5 assets by number of CM work orders.**

Save all three to `kpis.xlsx` in the run folder, one sheet each (`pd.ExcelWriter`).

**Prompt to try** (paste `ready.dtypes` and `ready.head()` first so Gemini knows the columns):
> Using this pandas DataFrame `ready`, calculate (a) counts by wo_type, (b) for closed CM rows, hours between
> date_raised and date_closed, with count, mean and median by asset_type, (c) the top 5 asset_id by number of CM
> rows. Save the three tables as sheets in one Excel file. Explain each step and tell me what you assumed.

**Check the AI's answer:**
* Did it filter to CM **and** closed before calculating time to restore?
* Did it parse the date columns as dates when loading the CSV? (If not, the subtraction fails or is wrong.)
* Do the counts in (a) add up to the number of KPI-ready rows?
* Why is median worth showing alongside mean for repair times?

In [ ]:
ready = pd.read_csv(run_dir / "work_orders_kpi_ready.csv", parse_dates=["date_raised", "date_closed"])

# TODO (a) by_type =

# TODO (b) restore = ...   (closed CM only; hours = (date_closed - date_raised).dt.total_seconds() / 3600)

# TODO (c) top5 =

# TODO save to run_dir / "kpis.xlsx" with three sheets

In [ ]:
xl_path = run_dir / "kpis.xlsx"
sheets = pd.read_excel(xl_path, sheet_name=None) if xl_path.exists() else {}
check(len(sheets) == 3 and sheets[list(sheets)[0]].iloc[:, -1].sum() == len(ready),
      "kpis.xlsx has three sheets and the type counts add up to the KPI-ready rows.",
      "Expected three sheets, with counts by type adding up to len(ready).")

## 6. The data note  [Level 3]

Write the note for the operations manager in the markdown cell below. Plain English, five to ten lines. Cover:

* What you did, in one or two sentences (not the code).
* How many records came in, how many are in the KPIs, how many were excluded and why.
* What Galway needs to fix.
* What you **assumed**, and what you **could not determine**. For example: "time to restore" here is raised to
  closed, which includes waiting for parts and for a technician. It is not hands-on repair time.

Then ask Gemini to critique your note: *"You are an operations manager. What in this note is unclear, and what
would you want to know that it does not say?"* Edit, but keep it yours.

*Your data note (double-click to edit):*

**Galway work orders, September 2026: data note**

## Stretch options

1. **Make KPIs part of the pipeline.** Move your KPI code into a `make_kpis(df, out_dir)` function in
   `pipeline.py`, called at the end of `run_pipeline`. Rerun both configs. Did anything break for Cork?
2. **Next month's export.** Ask Gemini to generate a small fake October file in Galway's format with one new
   problem it does not tell you about. Run the pipeline on it. Did the pipeline catch the problem, or did it get
   through? What rule would catch it?
3. **Sensor link.** Galway has CMP-003, PMP-002 and PMP-004 on condition monitoring. Using Module 06's outputs,
   which would you watch most closely and why?

## What you built today

You went from loading files without errors to a workflow that:

* loads different export formats through **configuration**, not code changes,
* standardises text, categories, numbers, units and dates, and **counts what it could not convert**,
* removes duplicates by a rule you can explain, and sends conflicts to a person,
* combines sources with audit columns and safe joins,
* checks quality with explicit rules, separates errors from warnings, and stops on structural problems,
* writes clean outputs, an issues list, a report and a log, every run, the same way.

And you used AI as a fast assistant whose work you checked, rather than as an oracle.